In [4]:
!pip install optuna
!pip install catboost

In [5]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split ,GroupShuffleSplit, cross_val_score,cross_val_predict, GroupKFold, StratifiedGroupKFold
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.pipeline import make_pipeline
from sklearn.pipeline import Pipeline as Pipe
from imblearn.pipeline import Pipeline as ImbPipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, FunctionTransformer
from sklearn.feature_selection import SelectFromModel
import optuna
from imblearn.over_sampling import SMOTE
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report, roc_auc_score
from sklearn.feature_selection import SelectKBest, mutual_info_classif
from sklearn.ensemble import VotingClassifier
from sklearn.decomposition import PCA
from sklearn.model_selection import ParameterGrid
from sklearn.metrics import recall_score
from sklearn.ensemble import AdaBoostClassifier

In [6]:
df = pd.read_csv('preprocessing.csv')
df.head()

,encounter_id,patient_nbr,race,gender,age,admission_type_id,discharge_disposition_id,admission_source_id,time_in_hospital,num_lab_procedures,...,change,diabetesMed,readmitted,medical_specialty_grouped,primary_diag,secondary_diag,additional_diag,med_change,num_med,service_utilization
0,2278392,8222157,1,0,1,5,18,1,1,41,...,0,0,0,Pediatrics,4,0,0,0,0,0
1,149190,55629189,1,0,2,1,1,7,3,59,...,1,1,0,Unknown,0,4,0,1,1,0
2,64410,86047875,2,0,3,1,1,7,2,11,...,0,1,0,Unknown,0,4,0,0,1,3
3,500364,82442376,1,1,4,1,1,7,2,44,...,1,1,0,Unknown,0,4,1,1,1,0
4,16680,42519267,1,1,5,1,1,7,1,51,...,1,1,0,Unknown,8,8,4,0,2,0


In [7]:
df.shape

(100111, 37)

In [8]:
numeric_cols = [
    'number_diagnoses', 'num_lab_procedures', 'num_med', 'time_in_hospital',
    'med_change', 'num_medications', 'number_outpatient', 'number_emergency',
    'number_inpatient', 'service_utilization', 'num_procedures'
]

categorical_cols = [
     'admission_type_id', 'discharge_disposition_id', 'admission_source_id',
    'max_glu_serum', 'A1Cresult', 'medical_specialty_grouped','primary_diag'
]


In [9]:
# Features
X = df.drop(columns=['readmitted','encounter_id'])  # drop target + identifiers

# Target
y = df['readmitted']

In [10]:
from sklearn.model_selection import GroupShuffleSplit

gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=15)

train_idx, test_idx = next(gss.split(X, y, groups=X['patient_nbr']))

X_train = X.iloc[train_idx]
X_test = X.iloc[test_idx]
y_train = y.iloc[train_idx]
y_test = y.iloc[test_idx]

groups = X_train['patient_nbr']

In [11]:
gss_val = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)

train_idx, val_idx = next(gss_val.split(X_train, y_train, groups=groups))

X_tr = X_train.iloc[train_idx]
X_val = X_train.iloc[val_idx]
y_tr = y_train.iloc[train_idx]
y_val = y_train.iloc[val_idx]

In [12]:
def evaluate_model(model):

    # Train

    '''if isinstance(model, CatBoostClassifier):
        model.fit(X_train.drop(columns=['patient_nbr']), y_train, cat_features=categorical_cols, verbose=0)
    else:'''
    model.fit(X_train.drop(columns=['patient_nbr']), y_train)
    # Predict
    y_pred = model.predict(X_test)

    # Accuracy
    acc = accuracy_score(y_test, y_pred)
    print("Accuracy:", acc)

    # Confusion matrix
    cm = confusion_matrix(y_test, y_pred)
    print("\nConfusion Matrix:\n", cm)

    # Classification report
    print("\nClassification Report:\n")
    print(classification_report(y_test, y_pred))

    # ROC-AUC
    if hasattr(model, "predict_proba"):
        y_proba = model.predict_proba(X_test)[:, 1]
        roc_auc = roc_auc_score(y_test, y_proba)
        print("\nROC-AUC:", roc_auc)

# Modeling

In [13]:

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_cols),
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_cols)
    ],
    remainder='passthrough'
)


## Decstion tree

In [ ]:
'''base_dt = DecisionTreeClassifier(
    max_depth=None,
    min_samples_split=2,
    random_state=15,
    class_weight='balanced'
)'''

base_dt = DecisionTreeClassifier(
    random_state=15,
    class_weight='balanced'
)

In [ ]:
#valiation
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import ParameterGrid
from sklearn.metrics import recall_score

In [ ]:
from sklearn.model_selection import ParameterGrid
from sklearn.metrics import recall_score

pipe_dt = ImbPipeline([
    ('preprocessor', preprocessor),
    ('model', base_dt)
])

param_grid = {
    'model__max_depth': [None, 5, 10, 20, 30],
    'model__min_samples_split': [2, 5, 10, 20],
    'model__min_samples_leaf': [1, 2, 5, 10]
}

Xtr = X_train.drop(columns=['patient_nbr'])
Xv = X_val.drop(columns=['patient_nbr'])

best_score = 0
best_params = None

for params in ParameterGrid(param_grid):

    pipe_dt.set_params(**params)

    pipe_dt.fit(X_tr.drop(columns=['patient_nbr']), y_tr)

    y_val_pred = pipe_dt.predict(X_val.drop(columns=['patient_nbr']))
    score = recall_score(y_val, y_val_pred)

    if score > best_score:
        best_score = score
        best_params = params

print("Best params:", best_params)
print("Best validation recall:", best_score)

Best params: {'model__max_depth': 5, 'model__min_samples_leaf': 10, 'model__min_samples_split': 2}
Best validation recall: 0.6279323513366066


### class weight

In [ ]:
pipe_dt.set_params(**best_params)

gkf = GroupKFold(n_splits=5)
scores = cross_val_score(
    pipe_dt,
    X_train.drop(columns=['patient_nbr']),
    y_train,
    groups=groups,
    cv=gkf,
    scoring='recall',
    n_jobs=-1
)

print("Fold recalls:", scores)
print(f'Mean recalls: {scores.mean():.3f}, Std: {scores.std():.3f}')
# Fit and predict
pipe_dt.fit(X_train.drop(columns=['patient_nbr']), y_train)
evaluate_model(pipe_dt)

Fold recalls: [0.60561798 0.59243006 0.63712844 0.60122019 0.5338843 ]
Mean recalls: 0.594, Std: 0.034
Accuracy: 0.6748028351801937

Confusion Matrix:
 [[12265  5416]
 [ 1099  1254]]

Classification Report:

              precision    recall  f1-score   support

           0       0.92      0.69      0.79     17681
           1       0.19      0.53      0.28      2353

    accuracy                           0.67     20034
   macro avg       0.55      0.61      0.53     20034
weighted avg       0.83      0.67      0.73     20034


ROC-AUC: 0.6584393489252186


## XGboost

In [ ]:
base_xgb = XGBClassifier(
    random_state=15,
    scale_pos_weight= len(y_train[y_train==0]) / len(y_train[y_train==1]),  # imbalance handling
    use_label_encoder=False,
    eval_metric='logloss',
    n_jobs=-1
)

In [ ]:
pipe_xgb= ImbPipeline([
    ('preprocessor', preprocessor),
    ('model', base_xgb)
])

param_grid_xgb = {
    'model__n_estimators': [400, 700, 1000],
    'model__max_depth': [4, 6, 8],
    'model__learning_rate': [0.0001,0.001,0.03],
    'model__subsample': [0.8, 1.0],
    'model__colsample_bytree': [0.8, 1.0],
    'model__min_child_weight': [1, 5]
}

best_score = 0
best_params = None


for params in ParameterGrid(param_grid_xgb):
    pipe_xgb.set_params(**params)
    pipe_xgb.fit(X_tr.drop(columns=['patient_nbr']), y_tr)
    y_val_pred = pipe_xgb.predict(X_val.drop(columns=['patient_nbr']))
    score = recall_score(y_val, y_val_pred)

    if score > best_score:
        best_score = score
        best_params = params

print("Best params:", best_params)
print("Best validation recall:", best_score)

/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning: [20:22:14] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning: [20:22:18] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning: [20:22:21] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning: [20:22:27] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/usr/local/lib/python3.12/dist-packages/xgboost/training.py:

Best params: {'model__colsample_bytree': 1.0, 'model__learning_rate': 0.001, 'model__max_depth': 4, 'model__min_child_weight': 1, 'model__n_estimators': 1000, 'model__subsample': 1.0}
Best validation recall: 0.6001091107474086


### Class weight

In [ ]:
pipe_xgb.set_params(**best_params)

# CV
gkf = GroupKFold(n_splits=5)
scores = cross_val_score(
    pipe_xgb,
    X_train.drop(columns=['patient_nbr']),
    y_train,
    groups=groups,
    cv=gkf,
    scoring='recall',
    n_jobs=-1
)

print("Fold recalls:", scores)
print(f'Mean recalls: {scores.mean():.3f}, Std: {scores.std():.3f}')

pipe_xgb.fit(X_train.drop(columns=['patient_nbr']), y_train)

evaluate_model(pipe_xgb)

Fold recalls: [0.61235955 0.61162918 0.63432417 0.59400998 0.61212121]
Mean recalls: 0.613, Std: 0.013


/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning: [20:47:44] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning: [20:47:54] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


Accuracy: 0.6219426974143956

Confusion Matrix:
 [[11013  6668]
 [  906  1447]]

Classification Report:

              precision    recall  f1-score   support

           0       0.92      0.62      0.74     17681
           1       0.18      0.61      0.28      2353

    accuracy                           0.62     20034
   macro avg       0.55      0.62      0.51     20034
weighted avg       0.84      0.62      0.69     20034


ROC-AUC: 0.6662104458643553


In [14]:
from lightgbm import LGBMClassifier

# Calculate scale_pos_weight like XGBoost
scale_pos_weight = len(y_train[y_train==0]) / len(y_train[y_train==1])

base_lgb = LGBMClassifier(
    random_state=15,
    class_weight=None,          # will use scale_pos_weight manually
    scale_pos_weight=scale_pos_weight,  # same as XGBoost
    n_jobs=-1,
    verbose=-1
)

param_grid_lgb = {
    'model__n_estimators': [400, 700, 1000],       # same as XGBoost
    'model__max_depth': [4, 6, 8],                 # max depth of trees
    'model__learning_rate': [0.0001,0.001,0.03],     # same concept as XGBoost
    'model__subsample': [0.8, 1.0],               # row subsampling (bagging_fraction in LGBM)
    'model__colsample_bytree': [0.8, 1.0],        # column subsampling (feature_fraction in LGBM)
    'model__min_child_weight': [1, 5]             # min_data_in_leaf in LGBM
}

pipe_lg= ImbPipeline([
    ('preprocessor', preprocessor),
    ('model', base_lgb)
])

best_score = 0
best_params = None


for params in ParameterGrid(param_grid_lgb):
    pipe_lg.set_params(**params)
    pipe_lg.fit(X_tr.drop(columns=['patient_nbr']), y_tr)
    y_val_pred = pipe_lg.predict(X_val.drop(columns=['patient_nbr']))
    score = recall_score(y_val, y_val_pred)

    if score > best_score:
        best_score = score
        best_params = params

print("Best params:", best_params)
print("Best validation recall:", best_score)

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/ut

Best params: {'model__colsample_bytree': 0.8, 'model__learning_rate': 0.03, 'model__max_depth': 4, 'model__min_child_weight': 5, 'model__n_estimators': 400, 'model__subsample': 0.8}
Best validation recall: 0.5870158210583742


### class weight

In [15]:
pipe_lg.set_params(**best_params)

# CV
gkf = GroupKFold(n_splits=5)
scores = cross_val_score(
    pipe_lg,
    X_train.drop(columns=['patient_nbr']),
    y_train,
    groups=groups,
    cv=gkf,
    scoring='recall',
    n_jobs=-1
)

print("Fold recalls:", scores)
print(f'Mean recalls: {scores.mean():.3f}, Std: {scores.std():.3f}')

pipe_lg.fit(X_train.drop(columns=['patient_nbr']), y_train)

evaluate_model(pipe_lg)

Fold recalls: [0.59157303 0.58310477 0.58833427 0.59400998 0.59559229]
Mean recalls: 0.591, Std: 0.004


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


Accuracy: 0.6401118099231307

Confusion Matrix:
 [[11389  6292]
 [  918  1435]]

Classification Report:

              precision    recall  f1-score   support

           0       0.93      0.64      0.76     17681
           1       0.19      0.61      0.28      2353

    accuracy                           0.64     20034
   macro avg       0.56      0.63      0.52     20034
weighted avg       0.84      0.64      0.70     20034



/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(



ROC-AUC: 0.6744623208977211


## Random Forest

In [ ]:
base_rf= RandomForestClassifier(
    random_state=15,
    class_weight='balanced'
)

In [ ]:


param_grid_rf = {
    'model__n_estimators': [100,200, 400],
    'model__max_depth': [5,10,15],
    'model__min_samples_split': [2, 5, 10],
    'model__min_samples_leaf': [1, 2, 5],}


pipe_rf = ImbPipeline([
    ('preprocessor', preprocessor),
    ('model', base_rf)
])


best_score = 0
best_params = None

for params in ParameterGrid(param_grid_rf):

    pipe_rf.set_params(**params)

    pipe_rf.fit(X_tr.drop(columns=['patient_nbr']), y_tr)

    y_val_pred = pipe_rf.predict(X_val.drop(columns=['patient_nbr']))
    score = recall_score(y_val, y_val_pred)

    if score > best_score:
        best_score = score
        best_params = params

print("Best params:", best_params)
print("Best validation recall:", best_score)

Best params: {'model__max_depth': 5, 'model__min_samples_leaf': 5, 'model__min_samples_split': 2, 'model__n_estimators': 400}
Best validation recall: 0.5815602836879432


### Class weight

In [ ]:
pipe_rf.set_params(**best_params)

gkf = GroupKFold(n_splits=5)
scores = cross_val_score(
    pipe_rf,
    X_train.drop(columns=['patient_nbr']),
    y_train,
    groups=groups,
    cv=gkf,
    scoring='recall',
    n_jobs=-1
)

print("Fold recalls:", scores)
print(f'Mean recalls: {scores.mean():.3f}, Std: {scores.std():.3f}')

pipe_rf.fit(X_train.drop(columns=['patient_nbr']), y_train)

evaluate_model(pipe_rf)


Fold recalls: [0.59550562 0.60449808 0.6174986  0.59123683 0.59393939]
Mean recalls: 0.601, Std: 0.010
Accuracy: 0.6308275930917441

Confusion Matrix:
 [[11202  6479]
 [  917  1436]]

Classification Report:

              precision    recall  f1-score   support

           0       0.92      0.63      0.75     17681
           1       0.18      0.61      0.28      2353

    accuracy                           0.63     20034
   macro avg       0.55      0.62      0.52     20034
weighted avg       0.84      0.63      0.70     20034


ROC-AUC: 0.6682996985366074


## imblearn ensemble

### ADA boost

In [ ]:

base_estimator = DecisionTreeClassifier(
        max_depth=5,                 # VERY important
        min_samples_leaf=5,          # helps minority stability
        class_weight='balanced',
        random_state=15,
)

base_ada = AdaBoostClassifier(
    estimator=base_estimator,
    random_state=15
)

In [ ]:

param_grid_ada = {
    'model__n_estimators': [50, 100, 200, 500],     # number of boosting rounds
    'model__learning_rate': [0.0001, 0.001, 0.03], # step size shrinkage
    'model__random_state': [15]                      # reproducibility
}

best_score = 0
best_params = None

pipe_ada= ImbPipeline([
    ('preprocessor', preprocessor),
    ('model', base_ada)
])



    # Fit model

for params in ParameterGrid(param_grid_ada):

    pipe_ada.set_params(**params)

    pipe_ada.fit(X_tr.drop(columns=['patient_nbr']), y_tr)

    y_val_pred = pipe_ada.predict(X_val.drop(columns=['patient_nbr']))
    score = recall_score(y_val, y_val_pred)

    if score > best_score:
        best_score = score
        best_params = params

print("Best params:", best_params)
print("Best validation recall:", best_score)

Best params: {'model__learning_rate': 0.001, 'model__n_estimators': 100, 'model__random_state': 15}
Best validation recall: 0.6279323513366066


### class weight

In [ ]:
pipe_ada.set_params(**best_params)

gkf = GroupKFold(n_splits=5)
scores = cross_val_score(
    pipe_ada,
    X_train.drop(columns=['patient_nbr']),
    y_train,
    groups=groups,
    cv=gkf,
    scoring='recall',
    n_jobs=-1
)

print("Fold recalls:", scores)
print(f'Mean recalls: {scores.mean():.3f}, Std: {scores.std():.3f}')

pipe_ada.fit(X_train.drop(columns=['patient_nbr']), y_train)  # train with raw data

evaluate_model(pipe_ada)

Fold recalls: [0.5494382  0.60998354 0.63039821 0.61397671 0.5338843 ]
Mean recalls: 0.588, Std: 0.038
Accuracy: 0.6419586702605571

Confusion Matrix:
 [[11486  6195]
 [  978  1375]]

Classification Report:

              precision    recall  f1-score   support

           0       0.92      0.65      0.76     17681
           1       0.18      0.58      0.28      2353

    accuracy                           0.64     20034
   macro avg       0.55      0.62      0.52     20034
weighted avg       0.83      0.64      0.71     20034


ROC-AUC: 0.6327866094960091


Bagging

In [ ]:
from sklearn.ensemble import BaggingClassifier
base_estimator = DecisionTreeClassifier(
    max_depth=5,
    min_samples_leaf=5,
    class_weight='balanced',
    random_state=15
)

# Bagging model
base_bagging = BaggingClassifier(
    estimator=base_estimator,
    random_state=15
)

# Parameter grid for Bagging
param_grid_bagging = {
    'model__n_estimators':[ 100, 200, 500],
    'model__max_samples': [0.5, 0.8, 1.0],
    'model__max_features': [0.5, 0.8, 1.0],
    'model__bootstrap': [True],
    'model__bootstrap_features': [False]
}

# Pipeline
pipe_bagging = ImbPipeline([
    ('preprocessor', preprocessor),
    ('model', base_bagging)
])

best_score = 0
best_params = None

# Tuning loop
for params in ParameterGrid(param_grid_bagging):
    pipe_bagging.set_params(**params)
    pipe_bagging.fit(X_tr.drop(columns=['patient_nbr']), y_tr)
    y_val_pred = pipe_bagging.predict(X_val.drop(columns=['patient_nbr']))
    score = recall_score(y_val, y_val_pred)

    if score > best_score:
        best_score = score
        best_params = params

print("Best params:", best_params)
print("Best validation recall:", best_score)

Best params: {'model__bootstrap': True, 'model__bootstrap_features': False, 'model__max_features': 0.8, 'model__max_samples': 1.0, 'model__n_estimators': 500}
Best validation recall: 0.5891980360065466


In [ ]:
# Apply the best parameters found during tuning
pipe_bagging.set_params(**best_params)

# Group K-Fold cross-validation
from sklearn.model_selection import GroupKFold, cross_val_score

gkf = GroupKFold(n_splits=5)
scores = cross_val_score(
    pipe_bagging,
    X_train.drop(columns=['patient_nbr']),
    y_train,
    groups=groups,
    cv=gkf,
    scoring='recall',
    n_jobs=-1
)

print("Fold recalls:", scores)
print(f'Mean recalls: {scores.mean():.3f}, Std: {scores.std():.3f}')

# Fit on full training data
pipe_bagging.fit(X_train.drop(columns=['patient_nbr']), y_train)

# Evaluate the model
evaluate_model(pipe_bagging)

Fold recalls: [0.58483146 0.59791552 0.6068424  0.59733777 0.59504132]
Mean recalls: 0.596, Std: 0.007
Accuracy: 0.6326245382849156

Confusion Matrix:
 [[11245  6436]
 [  924  1429]]

Classification Report:

              precision    recall  f1-score   support

           0       0.92      0.64      0.75     17681
           1       0.18      0.61      0.28      2353

    accuracy                           0.63     20034
   macro avg       0.55      0.62      0.52     20034
weighted avg       0.84      0.63      0.70     20034


ROC-AUC: 0.6691659019253551
